<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>

## RapidFire AI + Optuna: Adaptive SFT Hyperparameter Search

This tutorial shows how to use **RFOptuna** for Bayesian hyperparameter optimization
integrated into RapidFire's chunk-based training loop.

**Key difference from RFGridSearch / RFRandomSearch:**
- Grid/Random search decide all configs upfront
- **RFOptuna** uses Optuna's TPE sampler to suggest initial configs, then **prunes underperforming runs after each chunk** and replaces them with smarter suggestions

This notebook uses TinyLlama-1.1B-Chat with chat-formatted SFT, optimizing for **ROUGE-L** as the single objective metric.

### Enable Metric Loggers

In [ ]:
import os
os.environ["RF_MLFLOW_ENABLED"] = "true"

### Imports

Note: `RFOptuna` requires the `optuna` package. Install with:
```bash
pip install optuna
```

In [ ]:
from rapidfireai import Experiment
from rapidfireai.automl import (
    List,
    Range,
    RFOptuna,
    RFModelConfig,
    RFLoraConfig,
    RFSFTConfig,
)

### Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

train_dataset = dataset["train"].select(range(128))
eval_dataset = dataset["train"].select(range(128, 152))
train_dataset = train_dataset.shuffle(seed=42)
eval_dataset = eval_dataset.shuffle(seed=42)

### Define Data Processing Function

In [ ]:
def sample_formatting_function(row):
    """Function to preprocess each example from dataset"""
    SYSTEM_PROMPT = "You are a helpful and friendly customer support assistant. Please answer the user's query to the best of your ability."
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["instruction"]},
        ],
        "completion": [
            {"role": "assistant", "content": row["response"]}
        ]
    }

### Define Custom Eval Metrics Function

In [ ]:
def sample_compute_metrics(eval_preds):  
    """Optional function to compute eval metrics based on predictions and labels"""
    predictions, labels = eval_preds

    import evaluate
    rouge = evaluate.load("rouge")
    bleu = evaluate.load("bleu")

    rouge_output = rouge.compute(predictions=predictions, references=labels, use_stemmer=True)
    rouge_l = rouge_output["rougeL"]
    bleu_output = bleu.compute(predictions=predictions, references=labels)
    bleu_score = bleu_output["bleu"]

    return {
        "rougeL": round(rouge_l, 4),
        "bleu": round(bleu_score, 4),
    }

### Initialize Experiment

In [ ]:
experiment = Experiment(experiment_name="exp-optuna-chatqa-tiny1", mode="fit")

### Define Config Template with Search Space

Instead of listing every combination (GridSearch) or sampling blindly (RandomSearch),
we define a **search space** using `Range(...)` and `List([...])` inside a single
`RFModelConfig`. Optuna's TPE sampler will intelligently explore this space.

In [ ]:
config_template = RFModelConfig(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    peft_config=RFLoraConfig(
        r=List([8, 32]),
        lora_alpha=List([16, 32, 64, 128]),
        lora_dropout=0.1,
        target_modules=List([
            ["q_proj", "v_proj"],
            ["q_proj", "k_proj", "v_proj", "o_proj"],
        ]),
        bias="none",
    ),
    training_args=RFSFTConfig(
        learning_rate=List([1e-3, 1e-4]),
        lr_scheduler_type="linear",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        max_steps=128,
        gradient_accumulation_steps=1,
        logging_steps=2,
        eval_strategy="steps",
        eval_steps=4,
        bf16=True,
    ),
    model_type="causal_lm",
    model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
    formatting_func=sample_formatting_function,
    compute_metrics=sample_compute_metrics,
    generation_config={
        "max_new_tokens": 256,
        "temperature": 0.8,
        "top_p": 0.9,
        "top_k": 30,
        "repetition_penalty": 1.05,
    },
)

### Create RFOptuna Config Group

Key parameters:
- **`n_initial=4`**: Start with 4 configs (sampled by TPE)
- **`budget=6`**: Allow up to 6 total configs (including replacements for pruned runs)
- **`objective="maximize:rougeL"`**: Maximize ROUGE-L (from `sample_compute_metrics`)
- **`sampler="tpe"`**: Tree-structured Parzen Estimator (Bayesian)
- **`pruner="median"`**: Prune runs whose intermediate ROUGE-L falls below the median of peers

In [ ]:
config_group = RFOptuna(
    configs=[config_template],
    trainer_type="SFT",
    n_initial=4,
    budget=6,
    objective="maximize:rougeL",
    sampler="tpe",
    pruner="median",
    seed=42,
)

### Define Model Creation Function

In [ ]:
def sample_create_model(model_config):
    """Function to create model object for any given config; must return tuple of (model, tokenizer)"""
    from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM

    model_name = model_config["model_name"]
    model_type = model_config["model_type"]
    model_kwargs = model_config["model_kwargs"]

    if model_type == "causal_lm":
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    elif model_type == "seq2seq_lm":
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
    elif model_type == "masked_lm":
        model = AutoModelForMaskedLM.from_pretrained(model_name, **model_kwargs)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    return (model, tokenizer)

### Run Optuna-Powered Training

Behind the scenes:
1. `RFOptuna.get_runs()` asks Optuna's TPE sampler for 4 initial configs
2. RapidFire trains all 4 concurrently using chunk-based scheduling
3. After each chunk completes, the Optuna callback:
   - Resolves `rougeL` for each run
   - Uses median pruning: a run is pruned if its ROUGE-L falls below the median of peers
   - If pruned, suggests a replacement config (up to budget of 6)
4. RapidFire stops pruned runs and starts replacements automatically

In [ ]:
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,
    eval_dataset,
    num_chunks=4,
    seed=42)

### Inspect Optuna Study Results

After training, the Optuna study object is accessible on the `RFOptuna` instance.
`study.best_trial` returns the trial with the highest ROUGE-L score.

In [ ]:
import pandas as pd
from optuna.trial import TrialState

study = config_group._study

completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == TrialState.PRUNED]

print(f"Trials: {len(study.trials)} total, {len(completed)} completed, {len(pruned)} pruned")

try:
    best = study.best_trial
    print(f"Best trial: #{best.number} with rougeL = {best.value:.4f}")
except ValueError as exc:
    print(f"No best trial available yet ({exc}).")

trials_df = pd.DataFrame([
    {
        "trial": t.number,
        "state": t.state.name,
        "best": "✓" if t.number == study.best_trial.number else "",
        **{k.split(".")[-1]: v for k, v in t.params.items()},
        "rougeL": f"{t.value:.4f}" if t.value is not None else "—",
    }
    for t in study.trials
])
trials_df

### Get Results via RapidFire

In [ ]:
results = experiment.get_results()
results

### End Experiment

In [ ]:
experiment.end()

---

### Comparison: RFOptuna vs RFGridSearch vs RFRandomSearch

| Feature | RFGridSearch | RFRandomSearch | RFOptuna |
|---|---|---|---|
| Config selection | All combos upfront | Random sample upfront | Bayesian (TPE) — learns from results |
| Pruning | Manual via IC Ops | Manual via IC Ops | Automatic (Median / Hyperband pruner) |
| Replacement | Manual clone-modify | Manual clone-modify | Automatic — new suggestions within budget |
| Search space | `List([...])` | `List([...])`, `Range(...)` | `List([...])`, `Range(...)` |
| Best for | Small discrete spaces | Large spaces, no adaptation | Large spaces, adaptive exploration |

<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Thanks for trying RapidFire AI! ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>